---
title: "L7-05 · Identifiability before quantum causal learning"
short_title: "Identifiability before quantum causal learning"
subtitle: "A target is learnable only if every model that fits the accessible data gives it the same value; no estimator, however flexible, can do better, but quantum measurements in several bases can identify a causal direction that classical correlations cannot."
description: "Prove or disprove that a causal target is identifiable from a stated data map and model class, and repair a non-identified design with an intervention, a narrower model class, richer measurements, or bounds."
tags: [level-7, coursebook]
---

(lesson-l7-05)=

:::{div}
:class: qcai-meta
![Level 7 icon](../../_static/icons/qicon-level-7.svg) **Level 7 · Doctoral & postdoctoral** · about 120 minutes  
**Big idea.** A target is learnable only if every model that fits the accessible data gives it the same value; no estimator, however flexible, can do better, but quantum measurements in several bases can identify a causal direction that classical correlations cannot.  
**Before this lesson.** [L6-05](#lesson-l6-05), [L7-01](#lesson-l7-01)  
**Lab.** [Confounding and intervention](../../labs/03-causal-inference.ipynb)
:::

## What you will be able to do

- State identifiability as a property of a model class, a data map and a target, and prove that no estimator can be consistent for a non-identified target.
- Construct two models with identical observed statistics and different causal answers, for a classical structural model and for a quantum channel under restricted measurements.
- Derive the no-assumption bounds and the sensitivity bounds for an average treatment effect, and read off how much confounding would explain an effect away.
- Prove that same-basis Pauli correlations of any common-cause state lie in one tetrahedron and those of any direct cause in the dual tetrahedron, and decide when the data identify the direction.

## Why this matters

A large neural network trained on observational data from a hospital fits $P(X, Y)$ perfectly. Does the treatment work? The fit cannot say: below we write down two structural models with exactly the same $P(X, Y)$, one with an average effect of $0.6$ and one with none. More capacity, more data or a better optimizer cannot help, because the information is not in the data. That is a theorem, not a training failure.

Quantum causal learning inherits this problem and adds new twists. A device probed with one input and one measurement basis can hide whether it preserves coherence. Yet the same quantum theory also gives a surprise: correlations measured in several bases can reveal whether $A$ influenced $B$ or both came from a common source, which classical correlations alone never can [@ried2015]. This lesson makes both statements precise and proves them.

:::{dropdown} Key terms

**Data map**
: The function $D$ that sends a model to everything the experiment can observe, for example the joint distribution of recorded outcomes for the allowed preparations and measurements.

**Identifiable target**
: A quantity $T$ with $T(M_1) = T(M_2)$ whenever $D(M_1) = D(M_2)$, for all models $M_1, M_2$ in the class.

**Identified set**
: The set $\{T(M) : M \in \mathcal{M},\ D(M) = D_{\mathrm{obs}}\}$ of target values compatible with the data; a single point exactly when the target is identified at $D_{\mathrm{obs}}$.

**Sensitivity parameter**
: A number that bounds how strongly an untestable assumption may fail, here the confounding bound $\gamma$; the identified set is reported as a function of it.

**Pauli twirl**
: Averaging a state or channel over conjugation by Pauli operators; it keeps the same-basis correlations and removes everything else.

:::


In [1]:
import sys, pathlib, subprocess
here = pathlib.Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "qcai").is_dir()), None)
if root is None:  # for example on Google Colab: fetch the course's small teaching package
    root = pathlib.Path("quantum-causal-ai")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pozapas/quantum-causal-ai.git", str(root)], check=True)
    root = root.resolve()
sys.path.insert(0, str(root))
from qcai.viz import show as _show, show_mpl as _show_mpl
from qcai.lessonfigs import l7_05 as _lessonfigs

## Identification is a property of a model class

:::{prf:definition} Identifiability
:label: def-l7-05-ident
Fix a model class $\mathcal{M}$, a data map $D: \mathcal{M} \to \mathcal{D}$ and a target $T: \mathcal{M} \to \mathcal{T}$. The target is **identifiable** if $D(M_1) = D(M_2)$ implies $T(M_1) = T(M_2)$ for all $M_1, M_2 \in \mathcal{M}$; equivalently, $T = g \circ D$ for some function $g$ on the image of $D$. It is identifiable **at** $D_{\mathrm{obs}}$ if the identified set $\{T(M) : D(M) = D_{\mathrm{obs}}\}$ is a single point.
:::

Three ingredients, all of them choices. The model class encodes assumptions (a graph, no hidden variables, a channel family). The data map encodes the experiment (which variables are recorded, which interventions or measurement bases are allowed). The target is the scientific question. Changing any one of them can change the answer, which is why an identification statement must name all three [@pearlcausality; @bareinboim].

:::{prf:theorem} No estimator is consistent for a non-identified target
:label: thm-l7-05-noest
Suppose $M_1, M_2 \in \mathcal{M}$ have $D(M_1) = D(M_2)$ and real targets with $\Delta = |T(M_1) - T(M_2)| > 0$. Let $\hat T_N$ be any estimator, possibly randomized, computed from $N$ samples of the observable data. Then for every $N$,

$$\max_{i \in \{1, 2\}} \Pr_{M_i}\big[\,|\hat T_N - T(M_i)| \ge \Delta/2\,\big] \ge \tfrac12 .$$
:::

:::{prf:proof}
The samples have the same law under $M_1$ and $M_2$, so $\hat T_N$ has the same distribution under both. The events $A_i = \{|\hat T_N - T(M_i)| < \Delta/2\}$ are disjoint by the triangle inequality. Hence $\Pr(A_1) + \Pr(A_2) \le 1$ under that common distribution, so some $\Pr(A_i) \le 1/2$, and under $M_i$ the estimator misses by at least $\Delta/2$ with probability at least $1/2$. This holds for every $N$, so no estimator converges to the target under both models.
:::

:::{prf:remark} A prior chooses; it does not identify
A Bayesian posterior over $\mathcal{M}$ will report some value for $T$, but on a non-identified target the data cannot move the posterior within the set $\{M : D(M) = D_{\mathrm{obs}}\}$: the likelihood is constant there. The answer is the prior's. That is legitimate only if the prior is stated as an additional assumption, in the same way as a graph or a parametric form.
:::

## A classical pair: same data, different effects

Both models below have binary $X$ and $Y$ and exogenous noises that are independent Bernoulli variables. $\oplus$ is addition mod 2.

- **M1** (hidden common cause, no effect): $U \sim \mathrm{Bern}(1/2)$, $X := U \oplus N_X$ with $N_X \sim \mathrm{Bern}(0.1)$, $Y := U \oplus N_Y$ with $N_Y \sim \mathrm{Bern}(0.125)$.
- **M2** (direct effect, no confounding): $X \sim \mathrm{Bern}(1/2)$, $Y := X \oplus N_Y$ with $N_Y \sim \mathrm{Bern}(0.2)$.

In M1, $\Pr(U = 1 \mid X = 1) = 0.9$, so $\Pr(Y = 1 \mid X = 1) = 0.9 \cdot 0.875 + 0.1 \cdot 0.125 = 0.8$, and by symmetry $\Pr(Y = 1 \mid X = 0) = 0.2$. M2 gives the same numbers directly. So both produce $P(X, Y) = (0.4, 0.1, 0.1, 0.4)$ in the order $00, 01, 10, 11$.

$$
P_{M_1}(Y{=}1 \mid \mathrm{do}(X{=}x)) = \Pr(U \oplus N_Y = 1) = 0.5, \qquad P_{M_2}(Y{=}1 \mid \mathrm{do}(X{=}x)) = \begin{cases} 0.8 & x = 1 \\ 0.2 & x = 0 \end{cases}
$$ (eq-l7-05-do)

:::{figure} ../../_static/animations/l7_05_same_data.mp4
:alt: Two causal graphs side by side with identical bar charts of P(X, Y); after an intervention cuts the arrow into X in the first graph, the bars become 0.5 and 0.8.
:width: 100%

M1 and M2 give the same table $P(X, Y) = (0.4, 0.1, 0.1, 0.4)$. Graph surgery for $\mathrm{do}(X = 1)$ removes the arrow $U \to X$ in M1; the interventional answers are $0.5$ and $0.8$, and the average effects $0$ and $0.6$.
:::

:::{dropdown} What the animation shows
On the left, M1 has a dashed hidden node $U$ with arrows to $X$ and to $Y$. On the right, M2 has a single arrow $X \to Y$. Under each graph a bar chart shows $P(X, Y)$ over the outcomes 00, 01, 10, 11 with heights 0.4, 0.1, 0.1, 0.4; the two charts are identical, and a line states that no observational data can separate the models. Then $\mathrm{do}(X = 1)$ is applied: in M1 the arrow from $U$ into $X$ flashes red and disappears, and in both graphs $X$ is boxed as set by the experimenter. New bars show $P(Y = 1 \mid \mathrm{do}(X = 1))$: 0.5 in M1 and 0.8 in M2. The average treatment effects are $0.5 - 0.5 = 0$ and $0.8 - 0.2 = 0.6$.
:::

```{mermaid}
graph LR
  subgraph M1
    U((U hidden)) --> X1[X]
    U --> Y1[Y]
  end
  subgraph M2
    X2[X] --> Y2[Y]
  end
```

*The two graphs. Their observational distributions coincide; their intervention distributions do not, so the average effect is not identified from $P(X, Y)$ in a class that contains both.*

Enumerate both structural models exactly, confirm the identical observational table, compute the interventional answers, show that the naive contrast estimates $0.6$ whichever model produced the data, and evaluate the bounds of the next section.


In [2]:
import itertools
import numpy as np

def bern(p, v):
    return p if v == 1 else 1 - p

def m1(do_x=None):        # U ~ Bern(1/2), X := U xor N_X (0.1), Y := U xor N_Y (0.125)
    P = np.zeros((2, 2))
    for u, nx, ny in itertools.product((0, 1), repeat=3):
        x = (u ^ nx) if do_x is None else do_x
        P[x, u ^ ny] += 0.5 * bern(0.1, nx) * bern(0.125, ny)
    return P

def m2(do_x=None):        # X ~ Bern(1/2), Y := X xor N_Y (0.2)
    P = np.zeros((2, 2))
    for x0, ny in itertools.product((0, 1), repeat=2):
        x = x0 if do_x is None else do_x
        P[x, x ^ ny] += 0.5 * bern(0.2, ny)
    return P

print("observational P(X, Y), M1:\n", m1(), "\nM2:\n", m2(), "\nequal:", np.allclose(m1(), m2()))
for name, m in (("M1", m1), ("M2", m2)):
    ate = m(1)[1, 1] - m(0)[0, 1]
    print(f"{name}: P(Y=1|do(X=1)) = {m(1)[1, 1]:.2f},  ATE = {ate:.2f}")

# any estimator sees the same data law: the naive contrast converges to 0.6 under BOTH models
rng = np.random.default_rng(0)
P = m1().ravel()
draws = rng.choice(4, size=20000, p=P)          # identical draws would come from M2
x, y = draws // 2, draws % 2
print("naive contrast from 20000 samples:", round(y[x == 1].mean() - y[x == 0].mean(), 3), "(true ATE: 0 in M1, 0.6 in M2)")

def ate_bounds(gamma, px1=0.5, p1=0.8, p0=0.2):
    """Sharp bounds when |P(Y_x=1|X=1-x) - P(Y_x=1|X=x)| <= gamma; gamma = 1 gives the no-assumption bounds."""
    px0 = 1 - px1
    e1 = (px1 * p1 + px0 * max(0, p1 - gamma), px1 * p1 + px0 * min(1, p1 + gamma))
    e0 = (px0 * p0 + px1 * max(0, p0 - gamma), px0 * p0 + px1 * min(1, p0 + gamma))
    return round(e1[0] - e0[1], 10), round(e1[1] - e0[0], 10)

for g in (0, 0.2, 0.6, 0.8, 1):
    print(f"gamma = {g}: ATE in {ate_bounds(g)}")

observational P(X, Y), M1:
 [[0.4 0.1]
 [0.1 0.4]] 
M2:
 [[0.4 0.1]
 [0.1 0.4]] 
equal: True
M1: P(Y=1|do(X=1)) = 0.50,  ATE = 0.00
M2: P(Y=1|do(X=1)) = 0.80,  ATE = 0.60
naive contrast from 20000 samples: 0.59 (true ATE: 0 in M1, 0.6 in M2)
gamma = 0: ATE in (0.6, 0.6)
gamma = 0.2: ATE in (0.4, 0.8)
gamma = 0.6: ATE in (0.0, 0.8)
gamma = 0.8: ATE in (-0.2, 0.8)
gamma = 1: ATE in (-0.2, 0.8)


## Partial identification: what the data do pin down

Non-identified does not mean uninformative. Write $Y_x$ for the potential outcome under $\mathrm{do}(X = x)$, with **consistency**: if $X = x$ then $Y_x = Y$. Splitting on the observed treatment,

$$
P(Y_1{=}1) = P(Y{=}1, X{=}1) + P(Y_1{=}1, X{=}0), \qquad 0 \le P(Y_1{=}1, X{=}0) \le P(X{=}0).
$$ (eq-l7-05-split)

:::{prf:proposition} No-assumption bounds
:label: prop-l7-05-bounds
For binary $X, Y$ and any model consistent with $P(X, Y)$,

$$P(x_1, y_1) \le P(Y_1{=}1) \le P(x_1, y_1) + P(x_0), \qquad P(x_0, y_1) \le P(Y_0{=}1) \le P(x_0, y_1) + P(x_1),$$

and the resulting interval for the ATE has width exactly 1. The bounds are sharp. These are the no-assumption (natural) bounds of Manski (1990). For our table, $P(Y_1{=}1) \in [0.4, 0.9]$, $P(Y_0{=}1) \in [0.1, 0.6]$, and $\text{ATE} \in [-0.2, 0.8]$.
:::

:::{prf:proof}
The bounds follow from [](#eq-l7-05-split) and its analog for $Y_0$. The ATE interval is $[P(x_1,y_1) - P(x_0,y_1) - P(x_1),\ P(x_1,y_1) + P(x_0) - P(x_0,y_1)]$, of width $P(x_0) + P(x_1) = 1$. For sharpness, build a model on $(X, Y_0, Y_1)$ with the observed $P(X)$ and $P(Y_x{=}1 \mid X{=}x)$, and choose $P(Y_1{=}1 \mid X{=}0)$ and $P(Y_0{=}1 \mid X{=}1)$ freely in $[0, 1]$ with $Y_0, Y_1$ independent given $X$. Every such choice reproduces $P(X, Y)$ by consistency, and the extreme choices attain the endpoints.
:::

:::{figure} ../../_static/animations/l7_05_bounds.mp4
:alt: A derivation that turns into the interval from -0.2 to 0.8 on a number line with M1 at 0 and M2 at 0.6, then a sweep that shrinks the interval to the single point 0.6.
:width: 100%

Consistency plus the unseen term between 0 and $P(X = 0)$ gives $\text{ATE} \in [-0.2, 0.8]$. Bounding confounding by $\gamma$ shrinks the set; M1 leaves it once $\gamma < 0.6$.
:::

:::{dropdown} What the animation shows
The equation $P(Y_1 = 1) = P(Y_1 = 1, X = 1) + P(Y_1 = 1, X = 0)$ appears. By consistency the first term becomes the observed $P(Y = 1, X = 1) = 0.4$; the unseen second term lies between 0 and $P(X = 0) = 0.5$, so $P(Y_1 = 1) \in [0.4, 0.9]$. Likewise $P(Y_0 = 1) \in [0.1, 0.6]$, and the ATE lies in $[0.4 - 0.6, 0.9 - 0.1] = [-0.2, 0.8]$. A violet bar covers that interval on a number line; a red dot marks M1 at 0 and a cyan dot marks M2 at 0.6, both inside. Then the assumption that confounding shifts $P(Y_x = 1)$ between treatment groups by at most $\gamma$ is written, and $\gamma$ decreases from 1. The bar shrinks from both ends; at $\gamma = 0.6$ its left end reaches 0, where M1 sits, so explaining the effect away needs confounding of at least 0.6. At $\gamma = 0$ the bar is the single point 0.6.
:::

:::{prf:proposition} Sensitivity bounds
:label: prop-l7-05-sens
Assume $|P(Y_x{=}1 \mid X{=}1{-}x) - P(Y_x{=}1 \mid X{=}x)| \le \gamma$ for $x \in \{0, 1\}$. With $p_1 = P(y_1 \mid x_1)$ and $p_0 = P(y_1 \mid x_0)$, the sharp bounds are

$$P(Y_1{=}1) \in P(x_1)p_1 + P(x_0)\big[\max(0, p_1 - \gamma),\ \min(1, p_1 + \gamma)\big],$$

and symmetrically for $P(Y_0{=}1)$. For our table the ATE lies in $[\max(0.6 - \gamma, -0.2),\ \min(0.6 + \gamma, 0.8)]$. The value $0$ is inside exactly when $\gamma \ge 0.6$, and M1 has confounding $\gamma = 0.6$: $P(Y_1{=}1 \mid X{=}1) - P(Y_1{=}1 \mid X{=}0) = 0.8 - 0.2$.
:::


In [3]:
_show(_lessonfigs.sensitivity_bounds())

**Explore.** Slide the confounding bound $\gamma$. The identified set for the ATE (left) and its endpoints as functions of $\gamma$ (right). At $\gamma = 0$ the effect is identified as $0.6$; at $\gamma = 1$ it is the no-assumption interval.

:::{dropdown} What the figure shows
The left panel shows the identified set for the average treatment effect as a violet bar on an axis from $-0.35$ to $0.95$, with diamonds for M1 at 0 and M2 at 0.6. The right panel plots the lower bound $\max(0.6 - \gamma, -0.2)$ and the upper bound $\min(0.6 + \gamma, 0.8)$ against $\gamma$ from 0 to 1, with the region between shaded and a dashed line at the chosen $\gamma$. At $\gamma = 0$ the set is the point 0.6. The upper bound reaches 0.8 at $\gamma = 0.2$; the lower bound reaches 0 at $\gamma = 0.6$ and $-0.2$ at $\gamma = 0.8$, after which the set stays $[-0.2, 0.8]$.
:::

:::{tip} Three repairs, three different assumptions
**Intervene:** randomize $X$; the data map then contains $P(Y \mid \mathrm{do}(X))$ and the target is a function of it. **Measure more:** if $U$ is recorded and the graph of M1 is assumed, the backdoor formula $\sum_u P(y \mid x, u)P(u)$ identifies the effect (it gives $0.5$ for both $x$ in M1, with positivity since $P(x \mid u) \in \{0.1, 0.9\}$) [@pearlcausality]. **Narrow the class:** assume no confounding ($\gamma = 0$), which identifies $0.6$ but is an assumption the data cannot check. If none is available, report the interval.
:::

## A quantum pair: restricted measurements

:::{prf:example} Coherence hidden by the measurement basis
:label: ex-l7-05-dephase
Let the data map be: prepare $|0\rangle$, send it through the unknown qubit channel, measure $Z$. The identity channel and complete $Z$ dephasing $\mathcal{Z}(\rho) = \tfrac12(\rho + Z\rho Z)$ both output $|0\rangle$ and give outcome 0 with probability 1. The target "coherence transmission", say $\langle X\rangle$ at the output for input $|+\rangle$, is 1 for the identity and 0 for $\mathcal{Z}$. By Theorem [](#thm-l7-05-noest) no amount of $Z$ data on $|0\rangle$ can estimate it. Adding the input $|+\rangle$ and an $X$ measurement separates them: outcome $+$ with probability 1 versus $1/2$.
:::

:::{prf:proposition} Tomographically complete designs identify the channel
:label: prop-l7-05-tomo
If the input states $\{\rho_k\}$ span the space of operators on the input and the output measurements are informationally complete, then $D(\mathcal{E}_1) = D(\mathcal{E}_2)$ implies $\mathcal{E}_1 = \mathcal{E}_2$. For a qubit, the inputs $|0\rangle, |1\rangle, |+\rangle, |{+i}\rangle$ and measurements of $X$, $Y$, $Z$ suffice.
:::

:::{prf:proof}
Informational completeness means the measured statistics determine each output state $\mathcal{E}(\rho_k)$. A channel is linear, and the $\rho_k$ span the operator space, so its values on the $\rho_k$ determine it everywhere. For a qubit, $|0\rangle\langle0| \pm |1\rangle\langle1|$ give $I$ and $Z$, and $2|+\rangle\langle+| - I = X$, $2|{+i}\rangle\langle{+i}| - I = Y$ [@nielsenchuang].
:::

:::{prf:remark} The channel is identifiable; its dilation is not
Kraus operators $\{K_i\}$ and $\{\sum_j u_{ij}K_j\}$ with $u$ unitary define the same channel [@nielsenchuang]. Complete dephasing has both $\{I/\sqrt2, Z/\sqrt2\}$ and $\{|0\rangle\langle0|, |1\rangle\langle1|\}$. A target that depends on the decomposition, such as "which Kraus event happened", is therefore not identified by any input-output data, even tomographically complete data. This parallels classical counterfactuals, which interventional data alone do not fix ([L4-09](#lesson-l4-09)): such targets need additional model structure, stated as an assumption.
:::

## Quantum data can identify a causal direction

Two qubits $A$ and $B$ are measured in the same Pauli basis $\sigma_i \in \{X, Y, Z\}$, giving outcomes $a, b = \pm1$. Two model classes: a **common cause**, where $A$ and $B$ are halves of a shared state $\rho_{AB}$, so $c_i = \mathbb{E}[ab] = \operatorname{Tr}[\rho_{AB}\,\sigma_i\otimes\sigma_i]$; and a **direct cause** $A \to B$, where $A$ starts maximally mixed, is measured projectively, and its post-measurement eigenstate is sent through a channel $\mathcal{E}$ to $B$. Then $\Pr(a) = 1/2$, the state sent is $(I + a\sigma_i)/2$, and

$$
t_i = \mathbb{E}[ab] = \sum_{a = \pm1} \tfrac12\, a\, \operatorname{Tr}\Big[\sigma_i\, \mathcal{E}\big(\tfrac{I + a\sigma_i}{2}\big)\Big] = \tfrac12 \operatorname{Tr}\big[\sigma_i\, \mathcal{E}(\sigma_i)\big].
$$ (eq-l7-05-chan)

With $Z$ measurements only, the identity channel, the classical common cause $\tfrac12(|00\rangle\langle00| + |11\rangle\langle11|)$ and the entangled common cause $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt2$ all give $c_{zz} = 1$ and uniform marginals: classically, direction is not identified. Adding $X$ and $Y$ gives $(c_{xx}, c_{yy}, c_{zz}) = (1, 1, 1)$, $(0, 0, 1)$ and $(1, -1, 1)$. Since $\sum_i \sigma_i\otimes\sigma_i = 2F - I$ with $F$ the swap, every state has $c_{xx} + c_{yy} + c_{zz} = 2\operatorname{Tr}(\rho F) - 1 \le 1$, so the identity channel's sum of 3 cannot come from any common cause.

:::{figure} ../../_static/animations/l7_05_bases.mp4
:alt: Three columns of bars for an identity channel, a classical common cause and an entangled common cause; the Z bars are all +1, then X and Y bars differ, and their sums are 3, 1 and 1.
:width: 100%

In the $Z$ basis the three models agree. The $X$ and $Y$ correlations separate them, and $c_{xx} + c_{yy} + c_{zz} \le 1$ for every state rules out a common cause for the identity channel's pattern.
:::

:::{dropdown} What the animation shows
Three columns are labeled: a direct cause $A \to B$ through the identity channel, a classical common cause, and a common cause in the state $|\Phi^+\rangle$. First a $c_{zz}$ bar appears in each column, all of height $+1$, with the note that measuring $Z$ on both sides makes all three look the same. Then $c_{xx}$ bars appear: $+1$, $0$, $+1$. Then $c_{yy}$ bars: $+1$, $0$, $-1$. The sums are 3, 1 and 1. The bound $c_{xx} + c_{yy} + c_{zz} \le 1$ for every state, because the sum equals $2\operatorname{Tr}(\rho F) - 1$, closes the scene.
:::

The next theorem was proved in [L6-05](#lesson-l6-05). We restate it with a compact proof because the identification arguments below (the octahedron, the mixing-weight bound of the worked example) use its exact inequalities. Part (i) is the classical picture of Bell-diagonal states as a tetrahedron, whose separable members form the octahedron $|c_1| + |c_2| + |c_3| \le 1$ [@horodecki1996pra]; the use of the dual tetrahedron to infer causal direction from passive observation is the idea behind [@ried2015].

:::{prf:theorem} Two tetrahedra of correlation triples
:label: thm-l7-05-tetra
Within the two model classes above (a single common-cause state, or a single direct cause with maximally mixed $A$, projective Pauli measurement and eigenstate re-preparation):

(i) for every two-qubit state, $c = (c_1, c_2, c_3)$ satisfies $1 - c_1 - c_2 - c_3 \ge 0$, $1 - c_1 + c_2 + c_3 \ge 0$, $1 + c_1 - c_2 + c_3 \ge 0$, $1 + c_1 + c_2 - c_3 \ge 0$: the tetrahedron with vertices $(-1,-1,-1), (-1,1,1), (1,-1,1), (1,1,-1)$;

(ii) for every qubit channel, $t$ from [](#eq-l7-05-chan) satisfies the same inequalities with $t_2$ replaced by $-t_2$: the tetrahedron with vertices $(1,1,1), (1,-1,-1), (-1,1,-1), (-1,-1,1)$;

(iii) both sets are exactly attained, and their intersection is the octahedron $|c_1| + |c_2| + |c_3| \le 1$. An observed triple outside the octahedron identifies which class produced it; a triple inside it does not.
:::

:::{prf:proof}
(i) The Pauli twirl $\rho' = \tfrac14\sum_{k=0}^{3} (\sigma_k\otimes\sigma_k)\rho(\sigma_k\otimes\sigma_k)$, with $\sigma_0 = I$, is a state. Conjugation by $\sigma_k\otimes\sigma_k$ multiplies $\sigma_i\otimes\sigma_j$ by $s_k(i)s_k(j)$, where $s_k(i) = \pm1$ records whether $\sigma_k$ commutes with $\sigma_i$; the characters satisfy $\sum_k s_k(i)s_k(j) = 4\delta_{ij}$ for $i, j \in \{0, 1, 2, 3\}$. So the twirl keeps $\sigma_i\otimes\sigma_i$ and removes every other Pauli term: $\rho' = \tfrac14(I + \sum_i c_i\,\sigma_i\otimes\sigma_i)$ with the same $c$. Its eigenvectors are the Bell states, where $(XX, YY, ZZ)$ take the values $(1,-1,1)$ on $|\Phi^+\rangle$, $(-1,1,1)$ on $|\Phi^-\rangle$, $(1,1,-1)$ on $|\Psi^+\rangle$ and $(-1,-1,-1)$ on $|\Psi^-\rangle$, which gives the four eigenvalues $\tfrac14(1 + c_1 - c_2 + c_3)$ and so on. Positivity is the four inequalities. (ii) The twirled channel $\mathcal{E}'(\rho) = \tfrac14\sum_k \sigma_k\mathcal{E}(\sigma_k\rho\sigma_k)\sigma_k$ is a channel with the same $t$, it is unital, and by the same character sum $\mathcal{E}'(\sigma_i) = t_i\sigma_i$. Its Choi state $(\mathrm{id}\otimes\mathcal{E}')(|\Phi^+\rangle\langle\Phi^+|)$, starting from $|\Phi^+\rangle\langle\Phi^+| = \tfrac14(I + XX - YY + ZZ)$, equals $\tfrac14(I + t_1XX - t_2YY + t_3ZZ)$, a state with triple $(t_1, -t_2, t_3)$, so (i) applies. (iii) The vertices are the Bell states and the four Pauli unitary channels. The two systems of inequalities together are $\pm c_1 \pm c_2 \pm c_3 \le 1$ for all eight sign choices, which is the octahedron.
:::

:::{figure} ../../_static/animations/l7_05_tetrahedra.mp4
:alt: A rotating cube containing a blue tetrahedron, an amber tetrahedron pointing the other way, and a violet octahedron where they overlap, with three marked points.
:width: 100%

The correlation triples of all common-cause states (blue) and of all direct causes (amber) form dual tetrahedra in the cube $[-1, 1]^3$. Inside the violet octahedron $|c_1| + |c_2| + |c_3| \le 1$ the direction is not identified.
:::

:::{dropdown} What the animation shows
A wire cube spans $-1$ to $1$ on the axes $\langle XX\rangle$, $\langle YY\rangle$, $\langle ZZ\rangle$. A blue tetrahedron is drawn, with vertices at the four Bell-state triples; a key says it holds the correlations of any common-cause state. An amber tetrahedron follows, pointing the opposite way, with vertices at the triples of the four Pauli unitary channels; its key says any direct-cause channel $A \to B$. Their overlap, a violet octahedron, fades in with the formula $|c_1| + |c_2| + |c_3| \le 1$. Three points appear: the identity channel at $(1, 1, 1)$, a vertex of the amber tetrahedron outside the blue one; $|\Phi^+\rangle$ at $(1, -1, 1)$, a vertex of the blue tetrahedron outside the amber one; and the classical common cause at $(0, 0, 1)$, a vertex of the octahedron, compatible with both. The camera then circles the solids.
:::


In [4]:
_show(_lessonfigs.correlation_tetrahedra())

**Explore.** Rotate the solids and move an observed triple along the diagonal $(s, s, s)$, from the singlet at $s = -1$ to the identity channel at $s = 1$. The title reports which model class can explain it.

:::{dropdown} What the figure shows
A 3-D view of the cube $[-1, 1]^3$ with the blue tetrahedron of common-cause states, the amber tetrahedron of direct-cause channels and the violet edges of their octahedral overlap. Labeled points mark $|\Phi^+\rangle$ at $(1, -1, 1)$, the identity channel at $(1, 1, 1)$, the classical common cause at $(0, 0, 1)$ and the singlet at $(-1, -1, -1)$. A red marker moves along $(s, s, s)$ as the slider goes from $-1$ to $1$ in steps of $0.2$: for $s \le -0.4$ only a common cause fits, for $-0.2 \le s \le 0.2$ both fit and the direction is not identified, and for $s \ge 0.4$ only a direct cause fits. The exact switch points are $s = \pm 1/3$.
:::

Compute the triples of the three models, then check Theorem [](#thm-l7-05-tetra) on 3000 random two-qubit states and 3000 random qubit channels (from random isometries).


In [5]:
# Same-basis Pauli correlation triples: states versus channels
I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Z = np.diag([1, -1])
P3 = (X, Y, Z)

def c_state(rho):                    # (<XX>, <YY>, <ZZ>) of a two-qubit state
    return np.array([np.real(np.trace(rho @ np.kron(s, s))) for s in P3])

def c_channel(kraus):                # t_i = (1/2) Tr[s_i E(s_i)]: A maximally mixed, measured in s_i, eigenstate sent through E
    return np.array([0.5 * np.real(np.trace(s @ sum(K @ s @ K.conj().T for K in kraus))) for s in P3])

phi = np.array([1, 0, 0, 1]) / np.sqrt(2)
models = {"identity channel": c_channel([I2]),
          "classical common cause": c_state(np.diag([0.5, 0, 0, 0.5])),
          "Phi+ common cause": c_state(np.outer(phi, phi))}
for k, v in models.items():
    print(f"{k:24s} c = {v.round(3)}   sum = {v.sum():+.0f}")

state_ok = lambda c: min(1 - c[0] - c[1] - c[2], 1 - c[0] + c[1] + c[2], 1 + c[0] - c[1] + c[2], 1 + c[0] + c[1] - c[2]) >= -1e-9
chan_ok = lambda t: min(1 + t[0] + t[1] + t[2], 1 + t[0] - t[1] - t[2], 1 - t[0] + t[1] - t[2], 1 - t[0] - t[1] + t[2]) >= -1e-9

rng = np.random.default_rng(3)
def random_state():                  # random mixed two-qubit state (Ginibre)
    G = rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4)); R = G @ G.conj().T
    return R / np.trace(R)

def random_channel():                # random qubit channel from a random isometry into qubit x environment
    G = rng.normal(size=(8, 2)) + 1j * rng.normal(size=(8, 2)); V, _ = np.linalg.qr(G)
    return [V[2 * k:2 * k + 2, :] for k in range(4)]   # Kraus operators, sum K^dag K = I

S = [c_state(random_state()) for _ in range(3000)]
C = [c_channel(random_channel()) for _ in range(3000)]
print("all states in their tetrahedron:", all(map(state_ok, S)), "| max sum over states:", round(max(s.sum() for s in S), 3))
print("all channels in their tetrahedron:", all(map(chan_ok, C)))
print("identity channel compatible with any state?", state_ok(models["identity channel"]))
print("classical common cause compatible with a channel?", chan_ok(models["classical common cause"]))

identity channel         c = [1. 1. 1.]   sum = +3
classical common cause   c = [0. 0. 1.]   sum = +1
Phi+ common cause        c = [ 1. -1.  1.]   sum = +1


all states in their tetrahedron: True | max sum over states: 0.898
all channels in their tetrahedron: True
identity channel compatible with any state? False
classical common cause compatible with a channel? True


## A protocol for identification before learning

```{mermaid}
graph TD
  A[State target T] --> B[State model class M]
  B --> C[State data map D]
  C --> D{Two models with same data and different T?}
  D -->|none exist: proof| E[Estimate T with any good learner]
  D -->|found| F{Repair}
  F --> G[Add intervention or basis]
  F --> H[Narrow M with stated assumption]
  F --> I[Report identified set]
```

*Identification comes before architecture. A counterexample pair settles non-identification for that design; a proof is needed for identification.*

:::{warning} Mixtures of mechanisms enlarge the class
Theorem [](#thm-l7-05-tetra) compares a pure common cause with a pure direct cause. If the true process can be a probabilistic or coherent mixture of both, the model class is larger and points outside the octahedron no longer pin down a single mechanism; one can still bound the weight of each, as the worked example shows. Ried et al. report an experiment in which quantum correlations alone sometimes resolve the causal question; read it to see which model class they assume [@ried2015]. Coherent mixtures need the process formalism of [L6-07](#lesson-l6-07).
:::

## Worked example

:::{admonition} How much direct cause must there be?
:class: tip

A process is a probabilistic mixture: with probability $w$ a direct cause $A \to B$ through some channel, otherwise a common cause through some state. The observed triple is $c = (0.6, 0.6, 0.6)$. Find the smallest $w$ compatible with Theorem [](#thm-l7-05-tetra), and a decomposition attaining it.

1. The observed triple is linear in the mixture: $c = w\,t + (1 - w)\,c'$ with $t$ in the channel tetrahedron and $c'$ in the state tetrahedron.
2. Sum the coordinates. States satisfy $c'_1 + c'_2 + c'_3 \le 1$ and channels $t_1 + t_2 + t_3 \le 3$. So $1.8 \le 3w + (1 - w) = 1 + 2w$, giving $w \ge 0.4$.
3. Attain it: take $t = (1, 1, 1)$ (identity channel) and $c' = (c - 0.4\,t)/0.6 = (1/3, 1/3, 1/3)$. Check $c'$: $1 - 3\cdot\tfrac13 = 0$ and $1 + \tfrac13 - \tfrac13 + \tfrac13 > 0$, so it is a valid state triple (a Bell-diagonal state on the boundary of the tetrahedron).
4. Hence the identified set for $w$ is $[0.4, 1]$: the upper end is attained by $w = 1$ with $t = c$, which lies in the channel tetrahedron since $1 + 3(0.6) > 0$ and $1 + 0.6 - 1.2 > 0$.

**Result.** The data identify not $w$ but the interval $[0.4, 1]$: at least 40 percent direct cause. A point claim about $w$ would need a further assumption or a further experiment, such as an intervention on $A$.
:::

## Exercises

**1. (Core)** A learner trained on the $Z$-basis records of Example [](#ex-l7-05-dephase) reports the identity channel because of a minimum-noise prior. Is the coherence conclusion identified by the data? Specify an experiment that tests it.

:::{dropdown} Solution
No. The identity and complete dephasing give the same data, so the conclusion comes from the prior (Remark on priors). Prepare $|+\rangle$, send it through the device, and measure $X$: the identity gives $+$ with probability 1, complete dephasing gives $+$ with probability $1/2$. A few dozen shots separate them with high confidence.
:::

**2. (Core)** Compute the no-assumption ATE bounds for $P(X{=}1) = 0.3$, $P(Y{=}1 \mid X{=}1) = 0.9$, $P(Y{=}1 \mid X{=}0) = 0.5$.

:::{dropdown} Solution
$P(x_1, y_1) = 0.27$, $P(x_0, y_1) = 0.35$, $P(x_0) = 0.7$, $P(x_1) = 0.3$. So $P(Y_1{=}1) \in [0.27, 0.97]$ and $P(Y_0{=}1) \in [0.35, 0.65]$, and $\text{ATE} \in [0.27 - 0.65, 0.97 - 0.35] = [-0.38, 0.62]$, of width 1.
:::

**3. (Stretch)** Show that in M1 the backdoor formula with the recorded $U$ returns ATE $= 0$, and explain why it would fail if $N_X$ were absent ($X = U$).

:::{dropdown} Solution
$P(Y{=}1 \mid x, u) = P(U \oplus N_Y = 1 \mid U = u) = 0.875$ if $u = 1$ and $0.125$ if $u = 0$, for either $x$. So $\sum_u P(y_1 \mid x, u)P(u) = \tfrac12(0.875 + 0.125) = 0.5$ for both $x$ and the ATE is 0. If $X = U$, then $P(X = 1 \mid U = 0) = 0$, so $P(y \mid x_1, u_0)$ is undefined: positivity fails and the backdoor formula cannot be evaluated, even though $U$ is observed.
:::

**4. (Stretch)** Prove that complete dephasing has the two Kraus decompositions $\{I/\sqrt2, Z/\sqrt2\}$ and $\{|0\rangle\langle0|, |1\rangle\langle1|\}$, and give the unitary relating them.

:::{dropdown} Solution
$\tfrac12(\rho + Z\rho Z)$ keeps the diagonal of $\rho$ and kills the off-diagonal, as does $|0\rangle\langle0|\rho|0\rangle\langle0| + |1\rangle\langle1|\rho|1\rangle\langle1|$. With $u = \tfrac{1}{\sqrt2}\begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix}$: $\tfrac{1}{\sqrt2}(I/\sqrt2 + Z/\sqrt2) = (I + Z)/2 = |0\rangle\langle0|$ and $\tfrac{1}{\sqrt2}(I/\sqrt2 - Z/\sqrt2) = |1\rangle\langle1|$.
:::

**5. (Challenge)** Prove the character identity $\sum_{k=0}^{3} s_k(i)s_k(j) = 4\delta_{ij}$ used in Theorem [](#thm-l7-05-tetra), where $\sigma_k\sigma_i\sigma_k = s_k(i)\sigma_i$.

:::{dropdown} Solution
$s_k(0) = 1$ and $s_0(i) = 1$; for $i, k \ge 1$, $s_k(i) = 1$ if $k = i$ and $-1$ otherwise. If $i = j$ the sum is $\sum_k s_k(i)^2 = 4$. If $i = 0 \ne j$: $\sum_k s_k(j) = 1 + 1 - 1 - 1 = 0$. If $i, j \ge 1$ distinct: $k = 0$ gives $+1$, $k = i$ gives $s_i(j) = -1$, $k = j$ gives $-1$, and the third index gives $(-1)(-1) = +1$; total 0.
:::

**6. (Challenge)** Under Theorem [](#thm-l7-05-tetra) a common cause yields $c_{xx} + c_{yy} + c_{zz} \le 1$. Find the largest value of the same sum for a direct cause, the channels attaining it, and the largest value of $c_{xx} - c_{yy} + c_{zz}$ for each class. Which combination best detects an entangled common cause?

:::{dropdown} Solution
For channels $t_1 + t_2 + t_3 \le 3$, attained only by the identity channel (vertex $(1,1,1)$). For $c_1 - c_2 + c_3$: on the state tetrahedron the maximum over vertices is 3, at $|\Phi^+\rangle = (1, -1, 1)$; on the channel tetrahedron the vertices give $1, 1, -3, 1$, so the maximum is 1 (from the inequality $1 - t_1 + t_2 - t_3 \ge 0$). So $c_{xx} - c_{yy} + c_{zz} > 1$ rules out every direct cause and certifies a common cause; its maximum 3 is reached only by $|\Phi^+\rangle$.
:::

## Check your understanding

**1. What disproves identification of a target for a given design?**

- **A.** A wide confidence interval
- **B.** A low training loss
- **C.** Two allowed models with the same accessible data and different target values
- **D.** A larger neural model that fits better

:::{dropdown} Show the answer
**C. Two allowed models with the same accessible data and different target values**

If two permitted models give identical data but different targets, no estimator can be consistent for both (Theorem [](#thm-l7-05-noest)). Loss, model size and interval width are properties of an estimate, not of identifiability.
:::

**2. With no assumptions beyond consistency, how wide is the interval for the ATE of a binary treatment on a binary outcome?**

- **A.** Exactly 1
- **B.** Exactly 2
- **C.** It shrinks to 0 with enough data
- **D.** It depends on the sample size

:::{dropdown} Show the answer
**A. Exactly 1**

The unseen terms range over $[0, P(x_0)]$ and $[0, P(x_1)]$, so the width is $P(x_0) + P(x_1) = 1$, whatever the sample size.
:::

**3. The same-basis correlation triple is $(1, 1, 1)$. Within the classes of Theorem [](#thm-l7-05-tetra), what produced it?**

- **A.** Neither; the triple is impossible
- **B.** Either one; the data cannot tell
- **C.** A direct cause, since every state has a sum of at most 1
- **D.** A common cause, since only states reach the cube corners

:::{dropdown} Show the answer
**C. A direct cause, since every state has a sum of at most 1**

Every state satisfies $c_{xx} + c_{yy} + c_{zz} \le 1$, and $(1, 1, 1)$ is the identity channel's vertex of the channel tetrahedron.
:::

**4. In a DAG with $Z \to X$, $Z \to Y$ and $X \to Y$, with $Z$ observed, which formula gives $P(y \mid \mathrm{do}(x))$?** *(Review of [L4-04](#lesson-l4-04))*

- **A.** $P(y, x)/P(z)$
- **B.** $\sum_z P(y \mid x, z)\,P(z \mid x)$
- **C.** $P(y \mid x)$
- **D.** $\sum_z P(y \mid x, z)\,P(z)$

:::{dropdown} Show the answer
**D. $\sum_z P(y \mid x, z)\,P(z)$**

$Z$ blocks the only backdoor path, so adjustment averages the stratum-specific outcome over the marginal $P(z)$, not over $P(z \mid x)$.
:::

## Common misconceptions

- **"A flexible enough model will eventually learn the causal effect from observational data."** If two models in the class fit the data equally and disagree about the effect, Theorem [](#thm-l7-05-noest) shows no estimator can be consistent. Flexibility enlarges the class and can destroy identification.
- **"Quantum data always identify causal structure."** Only some triples do. Inside the octahedron $|c_1| + |c_2| + |c_3| \le 1$ both a state and a channel fit, and the classical common cause sits there. Restricting to one basis loses even that.
- **"If the channel is identified, everything about the process is."** Input-output data fix the channel but not its Kraus decomposition or environment; targets that depend on them need more structure or more access.

:::{caution} Scope and limits
Every statement here is relative to a model class and a data map; enlarging either can break identification, and narrowing the class is an assumption the data cannot check. The classical examples assume binary variables, consistency and, for the backdoor repair, a known graph and positivity. The quantum theorem assumes a single common-cause state or a single direct cause with maximally mixed $A$, projective Pauli measurements with eigenstate re-preparation, and same-basis correlations only; mixtures of mechanisms, other preparations, or noisy and miscalibrated measurements change the sets. The code checks are numerical illustrations of the proofs, not experiments.
:::

:::{seealso} Where this leads
Identification precedes the sample-complexity question of [L7-06](#lesson-l7-06): once a target is a function of the data, one can ask how many copies or shots estimate it to accuracy $\varepsilon$. The quantum part builds on informational access in [L6-05](#lesson-l6-05) and leads to the benchmark design of [L7-08](#lesson-l7-08), which must separate prediction from causal recovery.
:::

:::{dropdown} For instructors

**Why this lesson.** Research students often treat identification as an afterthought to model fitting. Putting a short impossibility theorem first, then classical and quantum counterexample pairs, makes the logical order explicit: target, class, data map, proof, then estimation [@pearlcausality; @bareinboim; @ried2015].

**Teaching notes.**

- Have students construct their own counterexample pair before showing M1 and M2; matching three numbers $P(X{=}1)$, $p_1$, $p_0$ is a short exercise.
- The no-assumption bounds are due to Manski (1990); the sensitivity version used here is one simple parametrization among several. Use the sensitivity interactive to show that reporting an identified set as a function of an assumption is more informative than a point estimate with an unstated assumption.
- Draw the two tetrahedra with physical models or the interactive figure; the duality (flip of the $YY$ axis) is easier to see than to read. Students met the theorem in [L6-05](#lesson-l6-05); spend the time on what is new here, the identification and partial-identification reading.
- Ask which repairs change the data map and which change the model class; the distinction matters for what a referee can check.

**Assessment.** The challenge on the tetrahedra tests whether students can use the inequalities as witnesses; the backdoor stretch exercise tests positivity; the worked example tests partial identification of a mixing weight.

**Going further.** Read the experiment of [@ried2015] and identify which assumptions of Theorem [](#thm-l7-05-tetra) it relaxes. Then compare with the quantum causal model frameworks of [@allen] and [@barrett], and formulate an identification question for a three-node quantum causal model.

:::

## References

This lesson draws on [@pearlcausality; @bareinboim; @ried2015; @horodecki1996pra; @nielsenchuang; @allen; @barrett; @costa].

**Further reading.**

- [@ried2015]: Experimental causal inference with quantum variables; shows that some quantum correlations reveal causal structure without intervention.
- [@costa]: Process-based quantum causal modeling, including Markov conditions that define the model classes used for identification.
- [@bareinboim]: Identification and the causal hierarchy in a modern causal AI textbook.

**Next:** [L7-06 · Sample complexity and resource accounting](#lesson-l7-06)
